# Cervical Cancer Stage Classification on Kaggle

This notebook launches the backend training script with Kaggle-friendly paths. It looks for the repository, finds the dataset, and writes checkpoints to the Kaggle working directory.

## Kaggle Run Guide

1. Open this notebook in Kaggle.
2. Make sure the repository is available in the session, or allow the notebook to clone it automatically.
3. If your dataset is mounted at a custom path, set `DATA_DIR` in Cell 3.
4. Run the notebook from top to bottom.
5. Check `/kaggle/working/Checkpoints` for `best_model.pt`, `last_model.pt`, `history.json`, and `metrics.json`.

Recommended defaults for the current run are already set in the training cell.

In [1]:
from __future__ import annotations

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/Shubh-Rawat7/Cervical-Cancer-Classifier.git'
REPO_NAME = 'Cervical-Cancer-Classifier'


def find_backend_dir() -> Path | None:
    search_roots = [Path('/kaggle/working'), Path('/kaggle/input'), Path.cwd()]
    direct_candidates = [
        Path('/kaggle/working/backend'),
        Path('/kaggle/input/backend'),
        Path.cwd() / 'backend',
        Path('/kaggle/working') / REPO_NAME / 'backend',
        Path('/kaggle/input') / REPO_NAME / 'backend',
    ]

    for candidate in direct_candidates:
        if (candidate / 'train.py').exists():
            return candidate

    for root in search_roots:
        if not root.exists():
            continue
        for match in root.rglob('train.py'):
            if match.name == 'train.py' and match.parent.name == 'backend':
                return match.parent
    return None


def clone_repo_if_needed() -> Path:
    work_root = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
    repo_dir = work_root / REPO_NAME
    backend_dir = repo_dir / 'backend'
    if (backend_dir / 'train.py').exists():
        return backend_dir

    if repo_dir.exists():
        print(f'Removing incomplete repo folder: {repo_dir}')
        subprocess.run(['rm', '-rf', str(repo_dir)], check=False)

    print(f'Cloning repository from {REPO_URL}')
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(repo_dir)], check=True)
    if not (backend_dir / 'train.py').exists():
        raise FileNotFoundError(f'Cloned repo but could not find backend/train.py in {backend_dir}')
    return backend_dir


BACKEND_DIR = find_backend_dir()
if BACKEND_DIR is None:
    BACKEND_DIR = clone_repo_if_needed()

REPO_ROOT = BACKEND_DIR.parent
os.chdir(REPO_ROOT)
if str(BACKEND_DIR) not in sys.path:
    sys.path.insert(0, str(BACKEND_DIR))

print('REPO_ROOT:', REPO_ROOT)
print('BACKEND_DIR:', BACKEND_DIR)
print('Python:', sys.executable)

Cloning repository from https://github.com/Shubh-Rawat7/Cervical-Cancer-Classifier.git
REPO_ROOT: d:\Cerivcal-Cancer-Stage-Classification\notebooks\Cervical-Cancer-Classifier
BACKEND_DIR: d:\Cerivcal-Cancer-Stage-Classification\notebooks\Cervical-Cancer-Classifier\backend
Python: c:\Users\asus\AppData\Local\Programs\Python\Python312\python.exe


In [2]:
CLASS_NAMES = ['Normal', 'CIN1', 'CIN2', 'CIN3', 'Cancer']


def _class_count(base: Path) -> int:
    try:
        return sum(1 for name in CLASS_NAMES if (base / name).is_dir())
    except Exception:
        return 0


def _has_class_folders(base: Path) -> bool:
    return base.exists() and base.is_dir() and _class_count(base) == len(CLASS_NAMES)


def _looks_like_dataset_root(base: Path) -> bool:
    if not base.exists() or not base.is_dir():
        return False

    if _has_class_folders(base):
        return True

    for split_name in ('train', 'val', 'test'):
        split_dir = base / split_name
        if _has_class_folders(split_dir):
            return True
    return False


def _root_score(root: Path) -> int:
    score = 0
    if _has_class_folders(root):
        score += 1000
    train_dir = root / 'train'
    val_dir = root / 'val'
    test_dir = root / 'test'
    if _has_class_folders(train_dir):
        score += 600
    if _has_class_folders(val_dir):
        score += 300
    if _has_class_folders(test_dir):
        score += 150
    return score


def find_data_dir() -> Path | None:
    raw_candidates = [
        os.environ.get('DATA_DIR', ''),
        '/kaggle/input/datasets/shubhrawat132/herlevdataset',
        '/kaggle/input/Herlev Dataset',
        '/kaggle/input/herlev-dataset',
        '/kaggle/input/herlevdataset',
        '/kaggle/input/cervical-cancer-stage-classification',
        '/kaggle/input/cervical-cancer-dataset',
        str(REPO_ROOT / 'Herlev Dataset'),
        str(REPO_ROOT / 'data'),
    ]

    best_root: Path | None = None
    best_score = -1

    for candidate_text in raw_candidates:
        if not candidate_text:
            continue
        candidate = Path(candidate_text)
        if not candidate.exists():
            continue

        search_roots = [candidate]
        try:
            search_roots.extend([p for p in candidate.rglob('*') if p.is_dir()])
        except Exception:
            pass

        for root in search_roots:
            if not _looks_like_dataset_root(root):
                continue
            score = _root_score(root)
            if score > best_score:
                best_score = score
                if _has_class_folders(root):
                    best_root = root
                elif _has_class_folders(root / 'train'):
                    best_root = root / 'train'
                elif _has_class_folders(root / 'val'):
                    best_root = root / 'val'
                elif _has_class_folders(root / 'test'):
                    best_root = root / 'test'

    return best_root


DATA_DIR = find_data_dir()
OUTPUT_DIR = Path('/kaggle/working/Checkpoints') if Path('/kaggle/working').exists() else (REPO_ROOT / 'backend' / 'Checkpoints')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if DATA_DIR is None:
    raise FileNotFoundError('Could not find the dataset. Set DATA_DIR to your Kaggle input folder and rerun this cell.')

print('DATA_DIR:', DATA_DIR)
print('OUTPUT_DIR:', OUTPUT_DIR)
print('Class folders found:', {name: (DATA_DIR / name).exists() for name in CLASS_NAMES})

FileNotFoundError: Could not find the dataset. Set DATA_DIR to your Kaggle input folder and rerun this cell.

In [ ]:
# Patch the cloned backend train script if it still uses hard-coded model.head references.
# This protects the Kaggle run from older upstream repo versions.

import re


def patch_train_script(train_path: Path):
    if not train_path.exists():
        print('train.py not found at', train_path)
        return

    text = train_path.read_text()
    patched = False

    if 'def _collect_param_groups(model, args):' not in text:
        helper_code = '''
def _collect_module_params(model, names):
    for name in names:
        module = getattr(model, name, None)
        if module is not None:
            return list(module.parameters())
    return []


def _collect_param_groups(model, args):
    groups = []
    backbone = getattr(model, 'backbone', None)
    if backbone is not None:
        backbone_params = [p for p in backbone.parameters() if p.requires_grad]
        if backbone_params:
            groups.append({'params': backbone_params, 'lr': args.lr_backbone})

    se_params = _collect_module_params(model, ('se',))
    head_params = _collect_module_params(model, ('head', 'classifier', 'fc'))

    if se_params:
        groups.append({'params': se_params, 'lr': args.lr_head})
    if head_params:
        groups.append({'params': head_params, 'lr': args.lr_head})
    if not groups:
        groups.append({'params': [p for p in model.parameters() if p.requires_grad], 'lr': args.lr_head})
    return groups
'''
        text = text.replace(
            'def load_checkpoint_state_dict(path: str | os.PathLike):',
            helper_code + '\n\ndef load_checkpoint_state_dict(path: str | os.PathLike):',
            1,
        )
        patched = True
        print(f'Inserted helper into {train_path}')

    phase2_block = '''
    # ── Phase 2: unfreeze last 3 blocks ──────────────────────────────────────
    p2 = args.phase2_epochs
    print("=" * 60)
    print(f"PHASE 2 — Unfreeze last 3 blocks ({p2} epochs, backbone_lr={args.lr_backbone:.2e})")
    print("=" * 60)

    model.load_state_dict(load_checkpoint_state_dict(best_ckpt_path))
    model.unfreeze_backbone(unfreeze_last_n_blocks=3)
    optimizer = optim.AdamW(
        _collect_param_groups(model, args),
        weight_decay=1e-4,
    )
    scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=max(p2, 1), eta_min=1e-7,
    )
    best_bal_acc, epoch_offset = run_phase(
        model, train_loader, val_loader, optimizer, scheduler,
        criterion, val_criterion, scaler, device,
        n_epochs=p2, best_bal_acc=best_bal_acc, patience=args.patience,
        best_ckpt_path=best_ckpt_path, history=history,
        epoch_offset=epoch_offset, mixup_alpha=args.mixup_alpha,
    )
    save_checkpoint(model, last_ckpt_path, {"phase": "phase2", "epoch": epoch_offset})
    _save_history()
'''

    phase2_pattern = re.compile(
        r'(?ms)^\s*# ── Phase 2: unfreeze last 3 blocks.*?_save_history\(\)\s*$',
    )
    text, count = phase2_pattern.subn(phase2_block, text, count=1)
    if count > 0:
        patched = True
        print(f'Replaced Phase 2 block in {train_path}')
    elif 'build_scheduler(' in text or 'model.head.parameters()' in text:
        text, count2 = re.subn(
            r'(?m)^\s*scheduler = build_scheduler\(optimizer, total_epochs=p2, warmup_epochs=min\(3, p2 // 4 \+ 1\)\)\s*$',
            '    scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(\n        optimizer, T_0=max(p2, 1), eta_min=1e-7,\n    )',
            text,
            count=1,
        )
        if count2 > 0:
            patched = True
            print(f'Replaced malformed scheduler line in {train_path}')

    if patched:
        train_path.write_text(text)
        print(f'Patched train.py: {train_path}')
    else:
        print(f'No changes made to {train_path}')


candidates = [BACKEND_DIR / 'train.py', Path('/kaggle/working/Cervical-Cancer-Classifier/backend/train.py')]
for candidate in candidates:
    patch_train_script(candidate)


In [ ]:
train_script = BACKEND_DIR / 'train.py'
command = [
    sys.executable,
    str(train_script),
    '--data-dir', str(DATA_DIR),
    '--output-dir', str(OUTPUT_DIR),
    '--epochs', '90',
    '--batch-size', '16',
    '--img-size', '256',
    '--backbone', 'tf_efficientnetv2_m',
    '--dropout', '0.30',
    '--lr-head', '3e-4',
    '--lr-backbone', '3e-5',
    '--phase1-epochs', '24',
    '--phase2-epochs', '26',
    '--mixup-alpha', '0.30',
    '--focal-gamma', '1.5',
    '--patience', '15',
    '--num-workers', str(min(4, os.cpu_count() or 2)),
]

print('Running:')
print(' '.join(command))
subprocess.run(command, check=True)

In [ ]:
artifacts = sorted(OUTPUT_DIR.glob('*'))
print('Training artifacts:')
for artifact in artifacts:
    print('-', artifact.name)

metrics_path = OUTPUT_DIR / 'metrics.json'
print('metrics.json exists:', metrics_path.exists())
if metrics_path.exists():
    print('metrics.json saved at', metrics_path)